# In Colab Analysis


In [ ]:
import os
# Make sure that you have the latest version of boto3
!pip install --upgrade boto3
!print(boto3.__version__) --> should be at least 1.28

# Import boto3 and json package
import boto3
import json
import google.colab
own_file, _ = google.colab.files.upload().popitem()
transcript = {'transcript': own_file}
# Define the access key and secret - insert your generated credentials here
access_key = os.environ["AWS_ACCESS_KEY_ID"]
access_secret = os.environ["AWS_SECRET_ACCESS_KEY"]

In [ ]:
import boto3
import json

# Define a function to call the bedrock API
def call_bedrock(prompt, assistant):
    # Call the bedrock client
    bedrock = boto3.client(service_name='bedrock-runtime',
                           region_name='us-east-1',
                           aws_access_key_id=access_key,
                           aws_secret_access_key=access_secret)

    # Tweak your preferred model parameters, prompt and assistant information
    body = json.dumps({
        "prompt": f"\n\nHuman:{prompt}\n\nAssistant:{assistant}",
        "max_tokens_to_sample": 500,
        "temperature": 0.1,
        "top_p": 0.7,})

    # Define the type of model that will be used
    modelId = 'anthropic.claude-v2'
    accept = 'application/json'
    contentType = 'application/json'

    # Call the Bedrock API
    response = bedrock.invoke_model(body=body, modelId=modelId, accept=accept, contentType=contentType)
    response_body = json.loads(response.get('body').read())
    print(response_body.get('completion'))
    body = str(response_body.get('completion')).split("\n")
    print(body[2:])

# Provide the instructions for analyzing the transcript
instructions = """
The factors of Aphasic speech that I need counted are:

Difficulty producing fluent, grammatical speech - speech is often slow, halting, with pauses as they search for word
Word retrieval issues - trouble thinking of specific words, use of filler words like um, circumlocution. May replace intended words with semantically similar ones (e.g. "cat" for "dog")
Grammatical errors - missing or incorrect function words like articles, prepositions, pronouns. Problems with verb tenses, conjugation, agreement. Sentences may be simplified
Repetitions and revisions - repeating or restating words, phrases or sentences due to trouble finding the right words or constructing the sentence
. Paraphasias - unintended substitutions of a word for another, including semantic paraphasias (related meaning) and phonemic paraphasias (similar sounds).
Neologisms - creating nonexistent "new" words
Perseveration - unintentionally repeating words or phrases over and over
Comprehension issues - trouble understanding complex sentences, fast speech. Rely more on context and cues

Provide the number of occurrences for each patient.

Estimate the western aphasia battery score. Present this in the format of "Estimated WAB Score: (insert number on scale of 0-100 here)"

Based on these values, how do you believe that our patient ranks in each of these activities against other Wernicke's aphasia patients, in terms of severity? Do not do this step if 0 occurrences are present in that factor. Express this severity as a percentile comparing against other Wernicke's Aphasia patients, in your professional opinion, supported by the data that you have collected. Provide an overall severity score


Synonyms or Describe the word in a sentence- for word finding difficulties
Use the specific severities to suggest therapies for each of the patients. State this in the form of "Because of the high prevalence of (insert factor of aphasic speech here), this patient should do (insert therapy technique here)." After that, describe each task in depth for a novice therapist and make these suggestions multi-step so it can be done as the patient recovers and help improve their speech.
The list below describes the typical progression of Wernicke's aphasia from its most severe form to mild impairment:
Stage 1: Very confused, with minimal understanding of written or spoken words. Speech is often garbled and nonsensical.
Stage 2: Confusion decreases, but speech is still garbled. Individuals may become frustrated from lack of understanding.
Stage 3: Survivors can now understand words in highly-contextual settings. For example, the survivor may understand when a family member gestures to a glass of water and asks if they want some.
Stage 4: Speech now has normal rhythm and intonation, but it may still not make sense. Comprehension has also improved a little at this point.
Stage 5: The survivor gains self-awareness, and now realizes that what they think they are saying is not always what they actually say. Individuals will also start to make more sense when speaking.
Stage 6: Survivors can now echo spoken words and read single words. They may also use writing to communicate more easily.
Stage 7: Speech begins to make sense and the individual uses real words appropriately. They might still struggle to find specific words, however.
Stage 8: Survivors may progress to very mild forms of aphasia, such as anomic aphasia. They may still struggle with finding the right words to say, but comprehension and speech coherency has greatly improved.
Provide the stage that the patient is at and suggest tools that the patients can use to improve their speech.
Here are some treatment techniques that may be used to address fluent aphasia:
Context-based therapy. This treatment technique focuses on improving comprehension through conversation. Speech therapists speak clearly, stressing important points through redundancy and non-verbal communication while the individual listens, taking an active role in their treatment by paraphrasing what has been said and asking for repetition as needed.
Comprehension exercises. Speech therapists may use a number of exercises focused on improving comprehension, gradually increasing the difficulty level as indicated. Exercises may include following simple directions, such as "point to your nose", or matching pictures with words.
Compensation techniques. Although relearning to grasp the meaning behind words is ideal, individuals may be encouraged to utilize compensation techniques to make up for their communication challenges. A common compensation technique is using non-verbal communication strategies, such as drawing a picture or gesturing.
Family training. Family members and caregivers can be a vital part of recovery. Speech therapists may provide family members with recommendations on how to promote effective communication, such as speaking slowly, clearly, and respectfully and discussing situationally-relevant information.
Non-invasive brain stimulation. Using transcranial direct-current stimulation and other non-invasive brain stimulation methods can promote neurological recovery, allowing individuals to regain communication skills. Although there are a number of studies on this treatment, it is still relatively new and is most effective when combined with other more conventional treatment methods.
Home programs. Continuing to practice receptive communication skills outside of therapy can boost recovery. This is because repetitive practice activates neuroplasticity, the brain's ability to rewire itself. While many speech therapists provide a list of exercises to practice at home, speech therapy apps, such as the CT Speech and Cognitive Therapy App, can be tailored to an individual's skill level to help them continue progressing toward recovery.
Suggest how to treat this patient based off of the symptoms you have seen and values you have calculated
"""

# Provide the cheat sheet for analyzing the transcript
cheat_sheet = """
TEMPLATE:
@Begin
@Languages:	eng
@Participants:	PAR Participant, INV Investigator
@ID:	eng|corpus|PAR|0;0.|male|Anomic||Participant||0.0|
@ID:	eng|corpus|INV|||||Investigator|||
@Media:	filename, video
@End
CHAT TRANSCRIPTION SYMBOL SUMMARY -- Abridged for AphasiaBank

Standard Template, change to fit each participant
@Begin
@Languages:	eng
@Participants:	PAR holland01a Participant, INV Margie Investigator
@ID:	eng|holland|PAR|71;02.|male|Anomic||Participant||86.8|
@ID:	eng|holland|INV|||||Investigator|||
@Media:	holland01a, video
@Date:	15-MAR-2006
@G:	task name (e.g., Sandwich)
*PAR:	transcript goes here.
@G:	task name (e.g., Umbrella)
@End

Note:  In Participant ID line, the info is:
Language|Corpus|Speaker|Age|Sex|WABtype||SpeakerRole||WAB-AQ|

Basic Utterance Terminators
.	period
?	question
!	exclamation

Special Utterance Terminators
+…	trailing off
+..?	trailing off of a question
+/.	interruption by another speaker
+/?	interruption of a question by another speaker
+//.	self-interruption
+//?	self-interruption of a question
+"/.	quotation follows on next line
+"	quoted utterance occurs on this line (use at beginning of utterance
	as link, not a terminator)
+<	lazy overlap marking (at beginning of utterance that overlapped the
	the previous utterance)


Utterance Termination Indicators
Syntax
Intonation
Pause
Semantics
Words
@n	neologism (e.g., sakov@n)
exclamations	common ones:  ah, aw, haha, ow, oy, sh, ugh, uhoh
interjections	common ones:  mhm, uhhuh, hm, uhuh
fillers	common ones:  &-um, &-uh
letters	s@l
letter sequence	abcdefg@k
xxx	unintelligible speech, not treated as a word
www		untranscribed material (e.g., looking through pictures, talking with
		spouse), must be followed by %exp tier (see below)
&+sounds	phonological fragment (&+sh &+w  we came home)

Scoped Symbols
[: text]	target/intended word for errors (e.g., tried [: cried])
[*]	error (e.g., paraphasia -- wɛk@u [: wet] [*])
[/]	retracing without correction (e.g., simple repetition)
	put repeated items between <> unless only one word was repeated
[//]	retracing with correction (e.g., simple word or grammar change)
	put changed items between <> unless only one word was changed

Local Events

&=text	simple local event and gestures (e.g., &=laughs, &=sighs,
	&=ges:fishing, &=head:shake, &=ges:wave)

Dependent Tiers
%exp:	must be used following www line to explain material not transcribed
	(e.g., %exp:    talks to spouse, %exp:    looks through pictures)

Other
()	shortenings -- e.g., runnin(g) for running, (be)cause for because
underscores	for combinations – e.g., you_know, a_lot, a_lot_of, how_come,
	kind_of, of_course, as_well, as_well_as, all_of_a_sudden,
	oh_my_gosh

Reminders
Use caps for proper nouns and the word "I" only; do not use caps at the beginning of utterances.
"""

# Format the answer
answer_format = """
Estimated WAB Score: (insert score here; Estimate the western aphasia battery score. Present this in the format of "Estimated WAB Score: (insert number on scale of 0-100 here)"
0–25: Very severe aphasia
26–50: Severe aphasia
51–75: Moderate aphasia
76–above: Mild aphasia

Difficulty producing fluent, grammatical speech: (number of occurrences), severity percentile
Word retrieval issues: (number of occurrences), severity percentile
Grammatical errors:  (number of occurrences), severity percentile
Repetitions and revisions: (number of occurrences), severity percentile
Paraphasias: (number of occurrences), severity percentile
Neologisms: (number of occurrences), severity percentile
Perseveration: (number of occurrences), severity percentile
Comprehension issues: (number of occurrences), severity percentile

Stage of progression: (insert stage number(1-8) over here), explain what this score means

Treatment Suggestions:
(insert suggestions in list over here)

Explanation for treatment options:
(over here)
"""

# Prepare the prompt for the Bedrock API
prompt = f"""
Use the following transcript for this analysis:

{transcript}

<instructions>
{instructions}
</instructions>

Use the cheat sheet below when analyzing the transcript given:
{cheat_sheet}

Format your answer like this:

{answer_format}
"""

# Call the Bedrock API with the prepared prompt
call_bedrock(prompt, "You are a speech therapist trying to diagnose and treat Wernicke's Aphasia. All you need to do is analyze the transcript using the data I give you.")


#Organize output into catagories for Streamlit


In [ ]:
import os
!pip install streamlit PyMuPDF
!pip install --upgrade boto3
!print(boto3.__version__) --> should be at least 1.28

access_key = os.environ["AWS_ACCESS_KEY_ID"]
access_secret = os.environ["AWS_SECRET_ACCESS_KEY"]
region = "us-east-1"

In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
import os
import streamlit as st
import boto3
import json
import pandas as pd
import plotly.express as px
import re
import logging
import fitz

# Configure logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

files = {"example": "/content/CASL-Sample-Test-Report.pdf"}

# Set page title and favicon
st.set_page_config(page_title="Speech Analysis", page_icon=":brain:")

# Set background color and font style
st.markdown(
    """
    <style>
    body {
        background-color: #f0f2f6;
        font-family: 'Arial', sans-serif;
    }
    </style>
    """,
    unsafe_allow_html=True
)

# Main pane
st.title("Personalized Speech Analysis")

access_key = os.environ["AWS_ACCESS_KEY_ID"]
access_secret = os.environ["AWS_SECRET_ACCESS_KEY"]
region = "us-east-1"

bedrock = boto3.client('bedrock-runtime',
                       aws_access_key_id=access_key,
                       aws_secret_access_key=access_secret,
                       region_name=region)

# Define the call_bedrock function

def read_pdf(file):
    if isinstance(file, str):
        document = fitz.open(file)
    else:
        document = fitz.open(stream=file.read(), filetype="pdf")

    text = ""
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        text += page.get_text()
    return text


example = read_pdf(files["example"])

def call_bedrock(prompt):
    try:
        body = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 4096,
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.3,
            "top_p": 0.9
        })

        modelId = 'anthropic.claude-3-5-sonnet-20240620-v1:0'
        response = bedrock.invoke_model(body=body, modelId=modelId, accept='application/json', contentType='application/json')
        response_body = json.loads(response.get('body').read())
        return response_body['content'][0]['text']
    except Exception as e:
        logger.error(f"Error in call_bedrock: {str(e)}")
        raise

# Provide the instructions for analyzing the transcript
instructions = """
The factors of speech that I need counted are:

Difficulty producing fluent, grammatical speech - speech is often slow, halting, with pauses as they search for word
Word retrieval issues - trouble thinking of specific words, use of filler words like um, circumlocution. May replace intended words with semantically similar ones (e.g. "cat" for "dog")
Grammatical errors - missing or incorrect function words like articles, prepositions, pronouns. Problems with verb tenses, conjugation, agreement. Sentences may be simplified
Repetitions and revisions - repeating or restating words, phrases or sentences due to trouble finding the right words or constructing the sentence
Neologisms - creating nonexistent "new" words
Perseveration - unintentionally repeating words or phrases over and over
Comprehension issues - trouble understanding complex sentences, fast speech. Rely more on context and cues

CASL-2 Speech and Language Analysis Framework
Evaluate each student using the following CASL-2 domains. For each domain, record observed patterns, challenges, and key errors. Use the observed data to estimate a CASL-2 standard score (mean = 100, SD = 15), corresponding percentile ranks, and performance levels (Below Average, Average, Above Average, etc.).

Domains to Analyze
Lexical/Semantic Skills

Key Subtests:
Antonyms: Ability to identify words with opposite meanings.
Synonyms: Ability to identify words with similar meanings.
Idiomatic Language: Understanding figurative expressions (e.g., “raining cats and dogs”).
Observations:
Note vocabulary diversity (specific vs. vague language).
Record word retrieval difficulties, incorrect word choices, reliance on fillers, or paraphasias.
Scoring:

Standard Score (mean = 100): (Estimated value based on observations)
Percentile Rank: (Estimate)
Performance Level: (e.g., Below Average, Average, Above Average)
Syntactic Skills

Key Subtests:
Sentence Expression: Ability to construct grammatically correct sentences from prompts.
Grammaticality Judgment: Ability to judge whether a sentence is grammatically correct.
Observations:
Identify grammatical errors (e.g., verb tense, subject-verb agreement).
Record simplified sentence structures or use of repetitive phrases.
Scoring:

Standard Score (mean = 100): (Estimated value based on observations)
Percentile Rank: (Estimate)
Performance Level: (e.g., Below Average, Average, Above Average)
Supralinguistic Skills

Key Subtests:
Inferences: Understanding implied meanings or reading between the lines.
Meaning from Context: Ability to derive word meanings from surrounding text.
Nonliteral Language: Interpretation of metaphors, idioms, and sarcasm.
Observations:
Assess ability to comprehend figurative language or implied meanings.
Record struggles with abstract or hypothetical reasoning.
Scoring:

Standard Score (mean = 100): (Estimated value based on observations)
Percentile Rank: (Estimate)
Performance Level: (e.g., Below Average, Average, Above Average)
Pragmatic Language Skills (if applicable)

Key Subtests:
Pragmatic Judgment: Appropriateness of responses in social scenarios.
Observations:
Record turn-taking, conversational rules, or adapting speech to context.
Scoring:

Standard Score (mean = 100): (Estimated value based on observations)
Percentile Rank: (Estimate)
Performance Level: (e.g., Below Average, Average, Above Average)
Final Summary
For each student, summarize the findings across all CASL-2 domains:

Lexical/Semantic Skills: Standard Score (X), Percentile Rank (X%), Performance Level (e.g., Below Average).
Syntactic Skills: Standard Score (X), Percentile Rank (X%), Performance Level (e.g., Average).
Supralinguistic Skills: Standard Score (X), Percentile Rank (X%), Performance Level (e.g., Above Average).
Overall Strengths and Weaknesses: Highlight areas of relative strength and areas requiring intervention.

Make sure to take this exmple into account. it is a good way to see what kinds of analysis need to be done..but make sure you are only doing what can be donefromt eh speech sample you are given:{example}


Here is what the diagnostic interpretation in the additional info sectionshould look like

Diagnostic Impressions
Jacob was referred for this evaluation to obtain further information with respect to his current development and skill level. Jacob was referred for this evaluation to obtain further information with respect to his [LIST - language, speech sound production, social pragmatics, vocabulary] skill(s).
The understanding and use of language are integrated into most learning tasks and facilitate Jacob's ability to effectively communicate and engage with others across his day. To examine Jacob's abilities, the CELF-5 was administered, and language index and subtest scores were obtained. Based upon the outcomes from the CELF-5, Jacob's language abilities are within expectations for his age. The outcomes of the various subtests suggest that he demonstrated the ability to comprehend, formulate, manipulate, and recall spoken sentences, follow directions, understand orally presented paragraphs as well as use grammatically correct words in sentences.
The outcomes from the CASL-2 suggest that Jacob demonstrated average general, receptive, expressive, lexical, syntactic, and supralinguistic language skills. With respect to his performance on the individual subtests, Jacob demonstrated consistently average abilities in all areas measured. He successfully identified and used vocabulary terms and word relationships, verbally produced grammatically correct sentences, described the meaning of unknown words based upon the context, and demonstrated higher-level language skills such as inferencing.




"""

# Provide the cheat sheet for analyzing the transcript
cheat_sheet = """
TEMPLATE:
@Begin
@Languages:	eng
@Participants:	PAR Participant, INV Investigator
@ID:	eng|corpus|PAR|0;0.|male|Anomic||Participant||0.0|
@ID:	eng|corpus|INV|||||Investigator|||
@Media:	filename, video
@End
CHAT TRANSCRIPTION SYMBOL SUMMARY -- Abridged for AphasiaBank

Standard Template, change to fit each participant
@Begin
@Languages:	eng
@Participants:	PAR holland01a Participant, INV Margie Investigator
@ID:	eng|holland|PAR|71;02.|male|Anomic||Participant||86.8|
@ID:	eng|holland|INV|||||Investigator|||
@Media:	holland01a, video
@Date:	15-MAR-2006
@G:	task name (e.g., Sandwich)
*PAR:	transcript goes here.
@G:	task name (e.g., Umbrella)
@End

Note:  In Participant ID line, the info is:
Language|Corpus|Speaker|Age|Sex|WABtype||SpeakerRole||WAB-AQ|

Basic Utterance Terminators
.	period
?	question
!	exclamation

Special Utterance Terminators
+…	trailing off
+..?	trailing off of a question
+/.	interruption by another speaker
+/?	interruption of a question by another speaker
+//.	self-interruption
+//?	self-interruption of a question
+"/.	quotation follows on next line
+"	quoted utterance occurs on this line (use at beginning of utterance
	as link, not a terminator)
+<	lazy overlap marking (at beginning of utterance that overlapped the
	the previous utterance)


Utterance Termination Indicators
Syntax
Intonation
Pause
Semantics
Words
@n	neologism (e.g., sakov@n)
exclamations	common ones:  ah, aw, haha, ow, oy, sh, ugh, uhoh
interjections	common ones:  mhm, uhhuh, hm, uhuh
fillers	common ones:  &-um, &-uh
letters	s@l
letter sequence	abcdefg@k
xxx	unintelligible speech, not treated as a word
www		untranscribed material (e.g., looking through pictures, talking with
		spouse), must be followed by %exp tier (see below)
&+sounds	phonological fragment (&+sh &+w  we came home)

Scoped Symbols
[: text]	target/intended word for errors (e.g., tried [: cried])
[*]	error (e.g., paraphasia -- wɛk@u [: wet] [*])
[/]	retracing without correction (e.g., simple repetition)
	put repeated items between <> unless only one word was repeated
[//]	retracing with correction (e.g., simple word or grammar change)
	put changed items between <> unless only one word was changed

Local Events

&=text	simple local event and gestures (e.g., &=laughs, &=sighs,
	&=ges:fishing, &=head:shake, &=ges:wave)

Dependent Tiers
%exp:	must be used following www line to explain material not transcribed
	(e.g., %exp:    talks to spouse, %exp:    looks through pictures)

Other
()	shortenings -- e.g., runnin(g) for running, (be)cause for because
underscores	for combinations – e.g., you_know, a_lot, a_lot_of, how_come,
	kind_of, of_course, as_well, as_well_as, all_of_a_sudden,
	oh_my_gosh

Reminders
Use caps for proper nouns and the word "I" only; do not use caps at the beginning of utterances.
"""

# Format the answer
answer_format = """
Do not mention ASD too much and instead


Difficulty producing fluent, grammatical speech: (number of occurrences), severity percentile
Word retrieval issues: (number of occurrences), severity percentile
Grammatical errors:  (number of occurrences), severity percentile
Repetitions and revisions: (number of occurrences), severity percentile
Neologisms: (number of occurrences), severity percentile
Perseveration: (number of occurrences), severity percentile
Comprehension issues: (number of occurrences), severity percentile

Evaluation of CASL Skills
Lexical/Semantic Skills: Standard Score (X), Percentile Rank (X%), Performance Level (e.g., Below Average).
Syntactic Skills: Standard Score (X), Percentile Rank (X%), Performance Level (e.g., Average).
Supralinguistic Skills: Standard Score (X), Percentile Rank (X%), Performance Level (e.g., Above Average).
Overall Strengths and Weaknesses: Highlight areas of relative strength and areas requiring intervention.


Other analysis/Best plans of action:
(insert suggestions in list over here)

Explanation:
(over here)

Additional Analysis:

"""

role = """You are a speech pathologist, a healthcare professional who specializes in evaluating, diagnosing, and treating communication disorders, including speech, language, cognitive-communication, voice, swallowing, and fluency disorders. Your role is to help patients improve their speech and communication skills through various therapeutic techniques and exercises.

Your are working with a student with speech impediments

The most important thing is that you stay kind to the child. Be constrctive and helpful rather than critical.

Each domain from the CASL-2 framework can be analyzed using the sample:

Lexical/Semantic Skills:

This category focuses on vocabulary knowledge, word meanings, and the ability to use words contextually. It measures both receptive and expressive language abilities related to word use.

Key Subtests:

Antonyms: Identifying words with opposite meanings.
Synonyms: Identifying words with similar meanings.
Idiomatic Language: Understanding and interpreting idioms and figurative language.

Evaluate vocabulary diversity (type-token ratio).
Note word-finding difficulties, incorrect word choices, or over-reliance on fillers (e.g., “like,” “stuff”).
Assess use of specific vs. vague language (e.g., "car" vs. "sedan").

Syntactic Skills:

This category evaluates understanding and use of grammar and sentence structure. It focuses on the ability to comprehend and produce grammatically correct sentences.

Key Subtests:

Sentence Expression: Producing grammatically correct sentences based on prompts.
Grammaticality Judgment: Identifying whether a sentence is grammatically correct.

Examine sentence structure for grammatical accuracy.
Identify errors in verb tense, subject-verb agreement, or sentence complexity.
Note the use of clauses, conjunctions, and varied sentence types.


Supralinguistic Skills:

This subcategory assesses higher-level language skills that go beyond literal meanings, such as understanding implied meanings, sarcasm, and complex verbal reasoning.

Key Subtests:

Inferences: Understanding information that is not explicitly stated.
Meaning from Context: Deriving meaning from surrounding text or dialogue.
Nonliteral Language: Interpreting figurative language, such as metaphors or irony

Look for use or understanding of figurative language, idioms, or humor.
Assess ability to handle ambiguous or implied meanings in context.
Identify advanced language use for abstract or hypothetical ideas.


Pragmatic Skills(focus less on this as it is not typically necessary for the age range you will be dealing with):
This category measures the ability to use language effectively in social contexts. It evaluates understanding of conversational rules, turn-taking, and adapting communication to different social situations.

Key Subtests:

Pragmatic Language Test: Assessing appropriateness of responses in social scenarios.

"""
# File uploader
uploaded_file = st.file_uploader("Choose a file", type=["txt", "rtf", "cha"])


if uploaded_file is not None:
    # Read the contents of the uploaded file
    file_content = uploaded_file.read().decode("utf-8")

    # Prepare the prompt for the Bedrock API
    prompt = f"""

    {role}

    Use the following transcript for this analysis:

    {file_content}

    <instructions>
    {instructions}

    Take the time stamps into account and reference them in the analysis
    </instructions>

    Use the cheat sheet below when analyzing the transcript given:
    {cheat_sheet}

    Format your answer like this:

    {answer_format}
    """

    # Call the call_bedrock function with the prompt
    response = call_bedrock(prompt)
    st.write(response)
    # Extract the values from the response
    lines = response.split('\n')
    data = {
        'Factor of Speech': [],
        'Occurrences': [],
        'Severity Percentile': []
    }
    treatment_suggestions = []
    treatment_explanations = []
    for line in lines:
        if ':' in line:
            factor, values = line.split(':', 1)
            values = values.split(',')
            if len(values) == 2:
                occurrences, percentile = values
                occurrences_match = re.findall(r'\d+', occurrences)
                percentile_match = re.findall(r'\d+', percentile)
                if occurrences_match and percentile_match:
                    occurrences = occurrences_match[0]  # Extract the numeric value from occurrences
                    percentile = percentile_match[0]  # Extract the numeric value from percentile
                    data['Factor of Speech'].append(factor.strip())
                    data['Occurrences'].append(int(occurrences))
                    data['Severity Percentile'].append(int(percentile))
        elif "Treatment Suggestions:" in line:
            treatment_suggestions = line.split("Treatment Suggestions:")[1].strip().split("\n")
        elif "Explanation for treatment options:" in line:
            treatment_explanations = line.split("Explanation for treatment options:")[1].strip().split("\n")

    # Ensure all arrays in the data dictionary have the same length
    max_length = max(len(arr) for arr in data.values())
    for key in data:
        data[key] = data[key][:max_length]

    # Create a DataFrame using the extracted data
    df = pd.DataFrame(data)

    # Display the DataFrame as a table in Streamlit
    st.subheader('Factors of Speech')
    st.table(df)

    # Create a plot using Plotly Express and display it in Streamlit
    fig = px.bar(df, x='Factor of Speech', y='Occurrences',
                 labels={'Factor of Speech': 'Factor of Speech', 'Occurrences': 'Occurrences'},
                 title='Occurrences of Factors of Speech')
    fig.update_xaxes(tickangle=-45)  # Rotate x-axis labels for better readability
    fig.update_layout(
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(size=14),
        title=dict(font=dict(size=20))
    )
    st.plotly_chart(fig)

    st.subheader("Additional Questions")
    question = st.text_input("Ask a question:")
    if question:
        # Prepare the prompt for the additional question
        question_prompt = f"""
        Based on the analysis response and treatment plan provided earlier, please answer the following question:
        You are a speech therapist answering additional questions about the analysis and treatment plan.
        Question: {question}
        """

        # Call the call_bedrock function with the question prompt
        question_response = call_bedrock(question_prompt,)

        # Display the response to the additional question
        st.write(question_response)

else:
    st.warning("Please upload a file for analysis.")